# f696 Stage1：完整窗口、同决策前状态对照

只读原 f696 A0 0bp Tiered 账户的 Qlib Recorder，SOXX 为最终基准。按窗口起点信号日已完成的 SOXX 5/20 日收益符号匹配好/差窗口；所有 819 个重叠窗口都保留。此分析只解释失败，不形成择时规则。

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

root = Path.cwd()
card = root / "research/stage1_f696_window_regime_pairing_20261023"
plan = json.loads((card / "plan.json").read_text())
source = plan["source"]
source_card = root / source["card"]
summary = json.loads((source_card / "summary.json").read_text())
assert summary["nas_readback"]["verified"] and summary["nas_readback"]["sha256"] == source["nas_verified_sha256"]
for artifact, expected in source["artifact_sha256"].items():
    path = source_card / "qlib_recorder" / source["experiment_id"] / source["recorder_id"] / "artifacts" / artifact
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, artifact
assert hashlib.sha256((root / source["qlib_provider"] / "qlib_store_manifest.json").read_bytes()).hexdigest() == source["qlib_provider_manifest_sha256"]
qlib.init(provider_uri=str((root / source["qlib_provider"]).resolve()), region="us", expression_cache=None, dataset_cache=None)
with R.uri_context(str(source_card / "qlib_recorder")):
    recorder = R.get_recorder(experiment_id=source["experiment_id"], recorder_id=source["recorder_id"])
    daily = recorder.load_object("analysis/daily")
    periods = recorder.load_object("analysis/periods")
    rolling = recorder.load_object("analysis/rolling126")
assert len(daily) == 944 and len(periods.loc[periods.kind.eq("quarter")]) == 16 and len(rolling) == 819
assert len(periods.loc[periods.kind.eq("year")]) == 4
partial = periods.loc[periods.kind.eq("year") & periods.partial_boundary_period, "period"].astype(str).tolist()
assert partial == ["2023", "2026"]
assert daily.trade_date.is_unique and daily.signal_date.is_unique
assert (daily[["paired_log_excess", "cash_opportunity_log", "SOXL_selection_log", "Stage1_direct_fees_log", "timing_and_benchmark_residual_log"]].notna().all().all())
closure = (daily.cash_opportunity_log + daily.SOXL_selection_log + daily.Stage1_direct_fees_log + daily.timing_and_benchmark_residual_log - daily.paired_log_excess).abs().max()
assert closure < 1e-12
calendar = pd.DatetimeIndex(D.calendar(start_time="2011-01-03", end_time="2026-10-09", freq="day"))
close = D.features(["SOXX"], ["$close"], start_time="2011-01-03", end_time="2026-10-09", freq="day")["$close"].droplevel("instrument").sort_index()
assert close.index.equals(calendar) and close.notna().all()
signal_by_trade = pd.Series(pd.DatetimeIndex(daily.signal_date), index=pd.DatetimeIndex(daily.trade_date))
start_trade = pd.DatetimeIndex(rolling.first_trade)
start_signal = pd.DatetimeIndex(signal_by_trade.reindex(start_trade))
assert start_signal.notna().all()
r5 = close.div(close.shift(5)).sub(1).reindex(start_signal)
r20 = close.div(close.shift(20)).sub(1).reindex(start_signal)
assert r5.notna().all() and r20.notna().all()
regime = np.select([((r5>=0)&(r20>=0)),((r5>=0)&(r20<0)),((r5<0)&(r20>=0)),((r5<0)&(r20<0))],
                   ["uptrend","rebound","pullback","decline"], default="unknown")
assert not (regime == "unknown").any()
x = rolling.copy()
x["start_state"] = regime
x["outcome"] = np.where(x.relative_log_excess < 0,"weak",np.where(x.relative_log_excess > 0,"good","tie"))
assert x.outcome.ne("tie").all()
def tally(z):
    return {"windows":int(len(z)),"cash_negative":int(z.cash_opportunity_log.lt(0).sum()),
            "selection_negative":int(z.SOXL_selection_log.lt(0).sum()),
            "both_negative":int((z.cash_opportunity_log.lt(0)&z.SOXL_selection_log.lt(0)).sum()),
            "mean_cash_log":float(z.cash_opportunity_log.mean()),
            "mean_selection_log":float(z.SOXL_selection_log.mean()),
            "mean_relative_log":float(z.relative_log_excess.mean()),
            "mean_cap_binding_fraction":float(z.cap_binding_fraction.mean())}
result={"experiment":plan["experiment"],"status":"read_only_exact_f696_window_regime_diagnostic",
        "source_recorder_id":source["recorder_id"],"source_nas_sha256":source["nas_verified_sha256"],
        "plan_sha256":hashlib.sha256((card / "plan.json").read_bytes()).hexdigest(),"partial_endpoint_years":partial,
        "daily_sessions":len(daily),"quarters":int(periods.kind.eq("quarter").sum()),"years_including_partial":int(periods.kind.eq("year").sum()),
        "rolling_windows":len(rolling),"daily_component_closure_max_abs":float(closure),
        "all_by_outcome":{name:tally(x[x.outcome.eq(name)]) for name in ["weak","good"]},
        "by_decision_time_start_state":{state:{name:tally(x[x.start_state.eq(state)&x.outcome.eq(name)]) for name in ["weak","good"]}
              for state in plan["fixed_definitions"]["regimes"]},
        "new_fits":0,"new_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "candidate_source_status":"SOXL same-field turnover source and first-public vintages unverified; no feature fit authorized",
        "independent_validation":False}
(card / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2)+"\n")
print(json.dumps({"all":result["all_by_outcome"],"by_start_state":result["by_decision_time_start_state"]},ensure_ascii=False,indent=2))

[73292:MainThread](2026-10-23 22:42:43,471) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[73292:MainThread](2026-10-23 22:42:43,472) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[73292:MainThread](2026-10-23 22:42:43,472) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{
  "all": {
    "weak": {
      "windows": 564,
      "cash_negative": 536,
      "selection_negative": 242,
      "both_negative": 214,
      "mean_cash_log": -0.11302686719209253,
      "mean_selection_log": 0.013235617725119336,
      "mean_relative_log": -0.10008283361816704,
      "mean_cap_binding_fraction": 0.3123944613306316
    },
    "good": {
      "windows": 255,
      "cash_negative": 94,
      "selection_negative": 70,
      "both_negative": 0,
      "mean_cash_log": 0.0004260265435131678,
      "mean_selection_log": 0.06812757903067107,
      "mean_relative_log": 0.06850095291969951,
      "mean_cap_binding_fraction": 0.23877995642701524
    }
  },
  "by_start_state": {
    "uptrend": {
      "weak": {
        "windows": 220,
        "cash_negative": 217,
        "selection_negative": 81,
        "both_negative": 78,
        "mean_cash_log": -0.10921038343868943,
        "mean_selection_log": 0.013160024341440632,
        "mean_relative_log": -0.09614980408196548,
     

## 中文结论

完整 Stage1 原账户的差窗口主要含现金机会损失，但同一决策前状态中的好窗口也可能出现现金拖累；SOXL 相对选择会抵消或加重它。不能从结果分组直接改现金权重或风险上限。下一个来源候选是前一完整交易日 SOXL 相对 SOXX 换手异常，它不同于原 39 项价格输入与已关闭的 SOXX 单边换手异常；SOXL 同口径历史、首次可用时间和 NAS 备份尚未核实，故当前不拟合、不跑账户。复用历史不是独立前向，SOXX 硬门未过。具体计数见 `result.json`。